In [1]:
import random
import numpy as np
import os

# 1. 파이썬 자체 난수 고정
random.seed(42)

# 2. 넘파이 난수 고정
np.random.seed(42)

# 3. 텐서플로우 사용하는 경우 (딥러닝)
import tensorflow as tf
tf.random.set_seed(42)

# 4. (선택) 실행 환경 내에서 다른 난수도 통제하고 싶을 때
os.environ['PYTHONHASHSEED'] = '42'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Conv2D, MaxPooling1D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, GridSearchCV ,cross_validate
from sklearn.metrics import *
import joblib

In [3]:
is_dynamic = {
    'STANDING': 0,
    'SITTING': 0,
    'LAYING': 0,
    'WALKING': 1,
    'WALKING_UPSTAIRS': 1,
    'WALKING_DOWNSTAIRS': 1
}

In [4]:
train_data = pd.read_csv('data01_train.csv')
test_data = pd.read_csv('data01_test.csv')

In [5]:
train_data.drop('subject', axis=1, inplace=True)
test_data.drop('subject', axis=1, inplace=True)

In [6]:
train_data['Activity_dynamic'] = train_data['Activity'].map(is_dynamic)
test_data['Activity_dynamic'] = test_data['Activity'].map(is_dynamic)

In [7]:
X_train = train_data.drop(['Activity', 'Activity_dynamic'], axis=1)
y_train = train_data.loc[:, 'Activity_dynamic']

In [8]:
X_test = test_data.drop(['Activity', 'Activity_dynamic'], axis=1)
y_test = test_data.loc[:, 'Activity_dynamic']

In [9]:
y_train.unique()

array([0, 1])

In [10]:
y_test.unique()

array([0, 1])

In [11]:
X_train.shape

(5881, 561)

## 학습 시작

### 1차원 데이터 형태

In [12]:
cnn = Sequential()

In [13]:
cnn.add(Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(561, 1)))
cnn.add(MaxPooling1D(pool_size=2, strides=2))

cnn.add(Conv1D(filters=32, kernel_size=3, activation='relu'))
cnn.add(MaxPooling1D(pool_size=2, strides=2))

cnn.add(Flatten())
cnn.add(Dense(units=128, activation='relu'))

cnn.add(Dense(units=1, activation='sigmoid'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 559, 32)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 279, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 277, 32)        │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 138, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4416)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       565,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 568,737 (2.17 MB)

 Trainable params: 568,737 (2.17 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [16]:
cnn.fit(X_train, y_train, batch_size=32, epochs=25)

Epoch 1/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9511 - loss: 0.1027
Epoch 2/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 4.6623e-04
Epoch 3/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 2.8644e-04
Epoch 4/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 1.0403e-04
Epoch 5/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 1.0386e-04
Epoch 6/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 7.0947e-05
Epoch 7/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 4.4969e-05
Epoch 8/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 2.8257e-05
Epoch 9/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 1.8767e-05
Epoch 10/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 1.3278e-05
Epoch 11/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 9.9330e-06
Epoch 12/25

In [17]:
cnn.save('cnn_model(1D)_static_vs_dynamic.h5')

In [18]:
y_pred = cnn.predict(X_test)

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


In [19]:
y_pred = (y_pred > 0.5).astype(int)

In [20]:
np.unique(y_pred)

array([0, 1])

In [21]:
print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 1.0
[[833   0]
 [  0 638]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       833
           1       1.00      1.00      1.00       638

    accuracy                           1.00      1471
   macro avg       1.00      1.00      1.00      1471
weighted avg       1.00      1.00      1.00      1471

Cohen's Kappa Score: 1.0000


## 2차원 데이터 형태

In [22]:
X_train = X_train.values

In [23]:
X_train

array([[ 0.28850798, -0.00919614, -0.10336219, ..., -0.60111976,
         0.33129814,  0.16516332],
       [ 0.26575678, -0.01657558, -0.09816338, ...,  0.34520491,
        -0.76918611, -0.14794449],
       [ 0.27870931, -0.01451131, -0.10871733, ..., -0.83356361,
         0.20243383, -0.03275467],
       ...,
       [ 0.26798135, -0.01834789, -0.10743994, ...,  0.62036305,
        -0.43724704, -0.57183996],
       [ 0.21278703, -0.04813008, -0.12100062, ..., -0.71800894,
         0.29285636,  0.0249202 ],
       [ 0.28381541, -0.04065785, -0.13801804, ..., -0.74024434,
         0.00834582, -0.14614344]])

In [24]:
X_train_reshaped = X_train.reshape(-1, 33, 17, 1)

In [25]:
X_train_reshaped.shape

(5881, 33, 17, 1)

In [26]:
cnn = Sequential()

In [27]:
cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same', input_shape=(33, 17, 1)))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

cnn.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same'))
cnn.add(MaxPooling2D(pool_size=2, strides=2))

cnn.add(Flatten())
cnn.add(Dense(units=128, activation='relu'))

cnn.add(Dense(units=1, activation='sigmoid'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 33, 17, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 8, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 8, 32)      │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 140,897 (550.38 KB)

 Trainable params: 140,897 (550.38 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [30]:
cnn.fit(X_train_reshaped, y_train, batch_size=32, epochs=25)

Epoch 1/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9269 - loss: 0.1652
Epoch 2/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 7.9956e-04
Epoch 3/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 5.4821e-04
Epoch 4/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 1.0000 - loss: 1.9525e-04
Epoch 5/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.0000 - loss: 2.3651e-04
Epoch 6/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.0000 - loss: 1.8815e-04
Epoch 7/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.0000 - loss: 1.1423e-04
Epoch 8/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 6.0906e-05
Epoch 9/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.0000 - loss: 3.5528e-05
Epoch 10/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.0000 - loss: 2.3615e-05
Epoch 11/25
184/184 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 1.0000 - loss: 1.8208e-05
Epoch 12/25


In [31]:
cnn.save('cnn_model(2D)_static_vs_dynamic.h5')

In [32]:
X_test.shape

(1471, 561)

In [33]:
X_test = X_test.values

In [34]:
X_test_reshaped = X_test.reshape(-1, 33, 17, 1)

In [35]:
X_test_reshaped.shape

(1471, 33, 17, 1)

In [36]:
y_pred = cnn.predict(X_test_reshaped)

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


In [37]:
y_pred = (y_pred > 0.5).astype(int)

In [38]:
np.unique(y_pred)

array([0, 1])

In [39]:
print('accuracy :',accuracy_score(y_test, y_pred))
print('='*60)
print(confusion_matrix(y_test, y_pred))
print('='*60)
print(classification_report(y_test, y_pred))
print(f"Cohen's Kappa Score: {cohen_kappa_score(y_test, y_pred):.4f}")

accuracy : 1.0
[[833   0]
 [  0 638]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       833
           1       1.00      1.00      1.00       638

    accuracy                           1.00      1471
   macro avg       1.00      1.00      1.00      1471
weighted avg       1.00      1.00      1.00      1471

Cohen's Kappa Score: 1.0000
